In [ ]:
"""
compare_fcoo_cmems.py

Quick-look diagnostic comparing surface-current forecasts from an FCOO/GETM
.nc file against a CMEMS .nc file: for each requested depth, plots the
magnitude difference (|V_fcoo| - |V_cmems|) and the direction difference
(bearing_fcoo - bearing_cmems, wrapped to [-180, 180] deg) on a common,
deliberately coarse lat/lon grid.

Design notes / assumptions (edit the CONFIG block to match your files):
  - FCOO grids are often curvilinear (2D lat/lon). CMEMS grids are usually
    regular (1D lat/lon). This script treats both the same way: it flattens
    each source grid to scattered (lon, lat, value) points, optionally
    subsamples them, and interpolates onto one common coarse target grid
    with scipy.interpolate.griddata. The "coarse" knob is GRID_RES_DEG; the
    "not too much computation" knob is SOURCE_SUBSAMPLE_MAX_POINTS (the cost
    of griddata is driven by the *source* point count, not the target grid,
    so that's the one that actually controls runtime).
  - Depth: both models are interpolated (in the vertical) onto the same
    TARGET_DEPTHS, since native vertical levels rarely match between models.
    Assumes depth coordinates are positive-down in metres; if a file stores
    depth as negative (height), the sign is flipped automatically.
  - Time: the FCOO file's first timestep is used as the reference valid
    time, and the nearest available CMEMS timestep is matched to it. Set
    TARGET_TIME explicitly if you want a specific valid time instead.
  - Direction is undefined when currents are near zero, so direction-diff
    pixels are masked out wherever either model's speed at that point is
    below MIN_SPEED_FOR_DIRECTION.

Run: python compare_fcoo_cmems.py
"""

import numpy as np
import xarray as xr
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ============================ CONFIG ============================
FCOO_FILE = "/home/ddyob/Documents/argo_piloting/model_tracker/data/fcoo_cache/dk_nested.velocities.Z3D_2026062412.nc"
CMEMS_FILE = "/home/ddyob/Documents/argo_piloting/model_tracker/data/"

# Candidate variable/coord names to auto-detect. Add your own if these
# don't match -- print(xr.open_dataset(path)) once to check.
U_NAMES = ["uo", "u", "uu", "water_u", "eastward_sea_water_velocity"]
V_NAMES = ["vo", "v", "vv", "water_v", "northward_sea_water_velocity"]
LAT_NAMES = ["latitude", "lat", "nav_lat"]
LON_NAMES = ["longitude", "lon", "nav_lon"]
DEPTH_NAMES = ["depth", "lev", "z", "deptho"]
TIME_NAMES = ["time", "Time", "valid_time"]

TARGET_DEPTHS = [0.0, 5.0, 20.0, 50.0]   # metres, positive down
GRID_RES_DEG = 0.1                        # coarse common output grid spacing
SOURCE_SUBSAMPLE_MAX_POINTS = 6000        # cap on source points per griddata call
MIN_SPEED_FOR_DIRECTION = 0.02            # m/s; below this, direction is masked
TARGET_TIME = None                        # None -> FCOO's first timestep
RANDOM_SEED = 0                           # for reproducible subsampling

OUTPUT_FILE = "fcoo_vs_cmems_diff.png"
# ==================================================================


def find_name(ds, candidates, kind):
    for name in candidates:
        if name in ds.variables or name in ds.coords:
            return name
    raise KeyError(
        f"Could not find a {kind} variable in {list(ds.variables)}. "
        f"Tried: {candidates}. Add the real name to the matching *_NAMES list."
    )


def get_lonlat(ds, lat_name, lon_name):
    """Return lon, lat as arrays broadcast to the same (possibly 2D) shape."""
    lat = ds[lat_name].values
    lon = ds[lon_name].values
    if lat.ndim == 1 and lon.ndim == 1:
        lon, lat = np.meshgrid(lon, lat)
    return lon, lat


def fix_depth_sign(da, depth_name):
    depth_vals = da[depth_name].values
    if np.nanmax(depth_vals) <= 0 and np.nanmin(depth_vals) < 0:
        da = da.assign_coords({depth_name: -depth_vals})
    return da


def interp_to_depths(da, depth_name, target_depths):
    """Interpolate onto target_depths, clamping requests outside the
    model's native depth range to the nearest available level (same
    out-of-bounds clamping pattern used elsewhere in the pipeline,
    rather than letting xarray return NaN for extrapolation)."""
    da = fix_depth_sign(da, depth_name)
    da = da.sortby(depth_name)
    d_min, d_max = float(da[depth_name].min()), float(da[depth_name].max())
    clamped = np.clip(target_depths, d_min, d_max)
    return da.interp({depth_name: clamped})


def regrid_layer(values, src_lon, src_lat, tlon, tlat, max_points, rng):
    values = np.asarray(values).ravel()
    src_lon = np.asarray(src_lon).ravel()
    src_lat = np.asarray(src_lat).ravel()

    good = np.isfinite(values) & np.isfinite(src_lon) & np.isfinite(src_lat)
    values, src_lon, src_lat = values[good], src_lon[good], src_lat[good]

    if values.size == 0:
        return np.full(tlon.shape, np.nan)

    if values.size > max_points:
        idx = rng.choice(values.size, size=max_points, replace=False)
        values, src_lon, src_lat = values[idx], src_lon[idx], src_lat[idx]

    pts = np.column_stack([src_lon, src_lat])
    out = griddata(pts, values, (tlon, tlat), method="linear")

    nan_mask = np.isnan(out)
    if nan_mask.any():
        out_nearest = griddata(pts, values, (tlon, tlat), method="nearest")
        out[nan_mask] = out_nearest[nan_mask]
    return out


def bearing_deg(u, v):
    """Compass bearing (deg clockwise from North) the current flows toward."""
    return (90.0 - np.degrees(np.arctan2(v, u))) % 360.0


def angle_diff(a, b):
    """Smallest signed difference a-b, wrapped to [-180, 180] deg."""
    return (a - b + 180.0) % 360.0 - 180.0


def main():
    rng = np.random.default_rng(RANDOM_SEED)

    fcoo = xr.open_dataset(FCOO_FILE)
    cmems = xr.open_dataset(CMEMS_FILE)

    u_name_f, v_name_f = find_name(fcoo, U_NAMES, "u"), find_name(fcoo, V_NAMES, "v")
    lat_name_f, lon_name_f = find_name(fcoo, LAT_NAMES, "lat"), find_name(fcoo, LON_NAMES, "lon")
    depth_name_f = find_name(fcoo, DEPTH_NAMES, "depth")
    time_name_f = find_name(fcoo, TIME_NAMES, "time")

    u_name_c, v_name_c = find_name(cmems, U_NAMES, "u"), find_name(cmems, V_NAMES, "v")
    lat_name_c, lon_name_c = find_name(cmems, LAT_NAMES, "lat"), find_name(cmems, LON_NAMES, "lon")
    depth_name_c = find_name(cmems, DEPTH_NAMES, "depth")
    time_name_c = find_name(cmems, TIME_NAMES, "time")

    # --- pick matching valid time ---
    ref_time = fcoo[time_name_f].values[0] if TARGET_TIME is None else np.datetime64(TARGET_TIME)
    fcoo_t = fcoo.sel({time_name_f: ref_time}, method="nearest")
    cmems_t = cmems.sel({time_name_c: ref_time}, method="nearest")
    actual_f_time = fcoo_t[time_name_f].values
    actual_c_time = cmems_t[time_name_c].values
    print(f"Reference time: FCOO={actual_f_time}  CMEMS={actual_c_time}")

    # --- vertical interpolation onto common target depths ---
    u_f_da = interp_to_depths(fcoo_t[u_name_f], depth_name_f, TARGET_DEPTHS)
    v_f_da = interp_to_depths(fcoo_t[v_name_f], depth_name_f, TARGET_DEPTHS)
    u_c_da = interp_to_depths(cmems_t[u_name_c], depth_name_c, TARGET_DEPTHS)
    v_c_da = interp_to_depths(cmems_t[v_name_c], depth_name_c, TARGET_DEPTHS)

    # --- horizontal coords ---
    lon_f, lat_f = get_lonlat(fcoo_t, lat_name_f, lon_name_f)
    lon_c, lat_c = get_lonlat(cmems_t, lat_name_c, lon_name_c)

    # --- common coarse target grid (intersection of both domains) ---
    lon_min = max(np.nanmin(lon_f), np.nanmin(lon_c))
    lon_max = min(np.nanmax(lon_f), np.nanmax(lon_c))
    lat_min = max(np.nanmin(lat_f), np.nanmin(lat_c))
    lat_max = min(np.nanmax(lat_f), np.nanmax(lat_c))
    if lon_min >= lon_max or lat_min >= lat_max:
        raise ValueError("FCOO and CMEMS domains do not overlap -- check lon/lat ranges.")

    target_lon = np.arange(lon_min, lon_max, GRID_RES_DEG)
    target_lat = np.arange(lat_min, lat_max, GRID_RES_DEG)
    tlon, tlat = np.meshgrid(target_lon, target_lat)
    print(f"Target grid: {tlon.shape[1]} x {tlon.shape[0]} cells at {GRID_RES_DEG} deg")

    n_depths = len(TARGET_DEPTHS)
    fig, axes = plt.subplots(
        n_depths, 2, figsize=(11, 3.6 * n_depths), squeeze=False
    )

    aspect = 1.0 / np.cos(np.radians(np.nanmean([lat_min, lat_max])))

    for di, depth in enumerate(TARGET_DEPTHS):
        u_f = regrid_layer(u_f_da.isel({depth_name_f: di}).values, lon_f, lat_f, tlon, tlat, SOURCE_SUBSAMPLE_MAX_POINTS, rng)
        v_f = regrid_layer(v_f_da.isel({depth_name_f: di}).values, lon_f, lat_f, tlon, tlat, SOURCE_SUBSAMPLE_MAX_POINTS, rng)
        u_c = regrid_layer(u_c_da.isel({depth_name_c: di}).values, lon_c, lat_c, tlon, tlat, SOURCE_SUBSAMPLE_MAX_POINTS, rng)
        v_c = regrid_layer(v_c_da.isel({depth_name_c: di}).values, lon_c, lat_c, tlon, tlat, SOURCE_SUBSAMPLE_MAX_POINTS, rng)

        mag_f, mag_c = np.hypot(u_f, v_f), np.hypot(u_c, v_c)
        mag_diff = mag_f - mag_c

        dir_f, dir_c = bearing_deg(u_f, v_f), bearing_deg(u_c, v_c)
        dir_diff = angle_diff(dir_f, dir_c)
        weak = (mag_f < MIN_SPEED_FOR_DIRECTION) | (mag_c < MIN_SPEED_FOR_DIRECTION)
        dir_diff = np.where(weak, np.nan, dir_diff)

        print(
            f"depth={depth:>5.1f} m | max|mag diff|={np.nanmax(np.abs(mag_diff)):.3f} m/s "
            f"| max|dir diff|={np.nanmax(np.abs(dir_diff)) if np.any(~np.isnan(dir_diff)) else float('nan'):.1f} deg"
        )

        # --- magnitude difference panel ---
        ax = axes[di, 0]
        vmax_mag = np.nanmax(np.abs(mag_diff))
        vmax_mag = vmax_mag if vmax_mag > 0 else 1e-3
        im = ax.pcolormesh(
            tlon, tlat, mag_diff, cmap="RdBu_r",
            norm=TwoSlopeNorm(vcenter=0, vmin=-vmax_mag, vmax=vmax_mag),
            shading="auto",
        )
        ax.set_aspect(aspect)
        ax.set_title(f"|V| diff (FCOO - CMEMS), depth={depth:g} m")
        ax.set_xlabel("lon")
        ax.set_ylabel("lat")
        fig.colorbar(im, ax=ax, label="m/s")

        # --- direction difference panel ---
        ax = axes[di, 1]
        im = ax.pcolormesh(
            tlon, tlat, dir_diff, cmap="RdBu_r",
            norm=TwoSlopeNorm(vcenter=0, vmin=-180, vmax=180),
            shading="auto",
        )
        ax.set_aspect(aspect)
        ax.set_title(f"direction diff (FCOO - CMEMS), depth={depth:g} m")
        ax.set_xlabel("lon")
        ax.set_ylabel("lat")
        fig.colorbar(im, ax=ax, label="deg")

    fig.tight_layout()
    fig.savefig(OUTPUT_FILE, dpi=150, bbox_inches="tight")
    print(f"Saved {OUTPUT_FILE}")


if __name__ == "__main__":
    main()